In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv.ipython import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [ ]:
agent = create_agent(
model=llm,
system_prompt= "You are a helpful assistant"
)

In [ ]:
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt="You are a helpful assistant"
)

In [ ]:
resp = agent.invoke(
input={
"messages":[

{"role":"user", "content":"Je m'appelle Yann"}
]
})

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
resp = agent.invoke(
input={
"messages":[

{"role":"user", "content":"Comment je m'appelle"}
]
})

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
basic_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
advenced_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [ ]:

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    env = request.runtime.context.get("env", "test")
    if env == "prod":
        model = advenced_llm
        print("advenced_llm selected")
    else:
        model = basic_llm
        print("basic_llm selected")
    return handler(request.override(model=model))

In [ ]:
agent2 = create_agent(
model=basic_llm,
tools=[],
middleware=[dynamic_model_selection],
debug=True
)

In [ ]:
resp=agent2.invoke(
input={"messages":[HumanMessage("C'est quoi un agent AI")]},
context={"env":"test"}
)

In [ ]:
from IPython.display import Markdown

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))

In [ ]:
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt="You are a helpful assistant",
)

In [ ]:
res=agent.invoke(input={"messages":[HumanMessage("je m'appelle Yann")]})

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
res=agent.invoke(input={"messages":[HumanMessage("C'est quoi mon nom")]})

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt="You are a helpful assistant",
    checkpointer=memory
)

In [ ]:
config = {"configurable":{"thread_id":"1"}}
res=agent.invoke(
    input={"messages":[HumanMessage("Je m'appelle Yann")]},
config=config
)

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
res=agent.invoke(
    input={"messages":[HumanMessage("comment je m'appelle")]},
    config=config)

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def get_weather(city : str):
    """"
    Get the weather of the given city
    """
    print("weather Tool invoked")
    return {
        "city":city,
        "teperature":23,
        "humidity":80,
        "pressure":120
    }

In [ ]:
@tool
def get_employee_info(employee_name : str):
    """"
    Get infos about the given employee (salary, seniority)
    """
    print("get_employee_info Tool invoked")
    return {
        "name":employee_name,
        "salary":34000,
        "seniority":5,
    }

In [ ]:
agent4 = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[get_weather, get_employee_info],
    checkpointer=memory,
    system_prompt="answer the user question using provided tools"
)

In [ ]:
config = {"configurable":{"thread_id":"1"}}
resp=agent4.invoke(input={'messages':[HumanMessage("La meteo a casablanca")]}, config=config)
print(resp['messages'][-1].content)

In [ ]:
config = {"configurable":{"thread_id":"1"}}
resp=agent4.invoke(input={'messages':[HumanMessage("Quel est le salaire de Yann")]}, config=config)
print(resp['messages'][-1].content)

In [ ]:
load_dotenv(override=True)

In [ ]:
from langchain_tavily import TavilySearch

In [ ]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

In [ ]:
@tool
def search_web(query:str):
    """Search for general web results"""
    print(f"search_web invoked with {query}")
    results = tavily.invoke({"query": query})
    return results

In [ ]:
agent5 = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[get_weather, get_employee_info, search_web],
    checkpointer=memory,
    system_prompt="answer the user question using provided tools"
)

In [ ]:
resp=agent5.invoke(
    input={"messages":[HumanMessage("Actualites sur le Burkina Faso")]},
    config=config
    )

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))